In [1]:
import numpy as np
import cv2
from skimage import exposure, filters


# -----------------------------
# Scaling / display
# -----------------------------
def compute_global_scale(stack, p_low=1, p_high=99):
    x = stack.astype(np.float32)
    lo, hi = np.percentile(x, (p_low, p_high))
    return float(lo), float(hi)

def to_uint8_fixed(img2d, lo, hi):
    img = exposure.rescale_intensity(img2d.astype(np.float32),
                                     in_range=(float(lo), float(hi)),
                                     out_range=(0, 255))
    return img.astype(np.uint8)

def to_uint8_auto(img2d, p_low=1, p_high=99):
    img = img2d.astype(np.float32)
    lo, hi = np.percentile(img, (p_low, p_high))
    lo, hi = float(lo), float(hi)
    if hi <= lo:
        return np.zeros_like(img2d, dtype=np.uint8)
    return to_uint8_fixed(img, lo, hi)

def draw_scale_bar(rgb, length_px=100, pad=12, thickness=4,
                   color=(255, 255, 255), loc="lower right"):
    H, W, _ = rgb.shape
    if loc == "lower right":
        x0 = W - pad - length_px
        y0 = H - pad
    elif loc == "lower left":
        x0 = pad
        y0 = H - pad
    else:
        raise ValueError("loc must be 'lower right' or 'lower left'")
    x1 = x0 + length_px
    y1 = y0 - thickness
    rgb[y1:y0, x0:x1] = color

def put_label(rgb, text, x=10, y=28):
    cv2.putText(rgb, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.75,
                (255, 255, 255), 2, cv2.LINE_AA)


# -----------------------------
# Bottom-percentile subtraction
# -----------------------------
def subtract_bottom_percentile_global(g, p=50, clamp=True):
    """
    Subtracts the p-th percentile of the frame (e.g. p=50 median).
    """
    g = g.astype(np.float32)
    b = float(np.percentile(g, float(p)))
    out = g - b
    if clamp:
        out[out < 0] = 0
    return out, b

def subtract_bottom_percentile_tilewise(g, tile=64, p=50, smooth_sigma_tiles=1.0, clamp=True):
    """
    Subtracts per-tile p-th percentile baseline, then smooths the baseline grid
    to avoid seams.
    """
    g = g.astype(np.float32)
    H, W = g.shape

    padH = (tile - (H % tile)) % tile
    padW = (tile - (W % tile)) % tile
    gp = np.pad(g, ((0, padH), (0, padW)), mode="reflect")
    Hp, Wp = gp.shape
    nH, nW = Hp // tile, Wp // tile

    base = np.zeros((nH, nW), dtype=np.float32)
    for i in range(nH):
        r0, r1 = i * tile, (i + 1) * tile
        for j in range(nW):
            c0, c1 = j * tile, (j + 1) * tile
            base[i, j] = np.percentile(gp[r0:r1, c0:c1], float(p))

    if smooth_sigma_tiles and smooth_sigma_tiles > 0:
        base = filters.gaussian(base, sigma=float(smooth_sigma_tiles), preserve_range=True)

    base_up = np.kron(base, np.ones((tile, tile), dtype=np.float32))[:Hp, :Wp]
    base_up = base_up[:H, :W]

    out = g - base_up
    if clamp:
        out[out < 0] = 0
    return out, base_up


# -----------------------------
# Video maker
# -----------------------------
def make_bottom_subtraction_video(
    green_stack, out_path="bottom_subtraction.avi",
    fps=10, k_start=0, k_end=None, stride=1, resize=1.0,
    scale_bar_px=100,
    raw_p_low=1, raw_p_high=99,

    # subtraction mode
    mode="global",          # "global" or "tile"
    bottom_p=50,            # e.g. 50 = median, 30 = darker baseline
    clamp=True,

    # tile options (mode="tile")
    tile=64,
    smooth_sigma_tiles=1.0,

    # show right panel scaling
    right_auto_scale=True   # True uses auto scale for visibility
):
    H, W, T = green_stack.shape
    if k_end is None:
        k_end = T

    raw_lo, raw_hi = compute_global_scale(green_stack[:, :, k_start:k_end:stride], raw_p_low, raw_p_high)

    outW, outH = int(W * resize), int(H * resize)
    outW2 = outW * 2

    fourcc = cv2.VideoWriter_fourcc(*"MJPG")
    writer = cv2.VideoWriter(out_path, fourcc, fps, (outW2, outH), True)
    if not writer.isOpened():
        raise RuntimeError("Could not open VideoWriter.")

    for k in range(k_start, k_end, stride):
        I = green_stack[:, :, k].astype(np.float32)

        if mode == "global":
            J, b = subtract_bottom_percentile_global(I, p=bottom_p, clamp=clamp)
            label = f"GLOBAL subtract p{bottom_p}={b:.1f}"
        elif mode == "tile":
            J, base = subtract_bottom_percentile_tilewise(
                I, tile=tile, p=bottom_p, smooth_sigma_tiles=smooth_sigma_tiles, clamp=clamp
            )
            label = f"TILE({tile}) subtract p{bottom_p}"
        else:
            raise ValueError("mode must be 'global' or 'tile'")

        # LEFT: raw fixed scaling
        raw8 = to_uint8_fixed(I, raw_lo, raw_hi)
        left = np.stack([raw8, raw8, raw8], axis=-1)
        draw_scale_bar(left, length_px=scale_bar_px)
        put_label(left, f"RAW fixed  k={k}", x=10, y=28)

        # RIGHT: subtracted
        if right_auto_scale:
            right8 = to_uint8_auto(J, 1, 99.7)
        else:
            # or show with same scaling as raw (often looks dark)
            right8 = to_uint8_fixed(J, raw_lo, raw_hi)

        right = np.stack([right8, right8, right8], axis=-1)
        draw_scale_bar(right, length_px=scale_bar_px)
        put_label(right, label, x=10, y=28)

        frame = np.concatenate([left, right], axis=1)

        if resize != 1.0:
            frame = cv2.resize(frame, (outW2, outH), interpolation=cv2.INTER_AREA)

        writer.write(frame[:, :, ::-1])  # RGB->BGR

        if (k - k_start) % (25 * stride) == 0:
            print(f"frame {k}/{k_end-1}")

    writer.release()
    print("Wrote:", out_path)


# -----------------------------
# Example run
# -----------------------------
if __name__ == "__main__":
    roi = 5
    data = np.load(f"./stacks_exp_148/exp_148_roi_{roi}_stack.npz")
    green_stack = data["Tstack"].astype(np.float32)  # (H,W,T)

    # 1) global subtract bottom 50%
    make_bottom_subtraction_video(
        green_stack,
        out_path=f"exp148_roi{roi}subtract_global_p50.avi",
        fps=10,
        stride=2,
        resize=0.5,
        mode="global",
        bottom_p=50,
        clamp=True,
        right_auto_scale=True
    )

    # 2) tilewise subtract bottom 50% (handles bias)
    make_bottom_subtraction_video(
        green_stack,
        out_path=f"exp148_roi{roi}_subtract_tile_p50.avi",
        fps=10,
        stride=2,
        resize=0.5,
        mode="tile",
        bottom_p=50,
        tile=64,
        smooth_sigma_tiles=1.2,
        clamp=True,
        right_auto_scale=True
    )


frame 0/68
frame 50/68
Wrote: exp148_roi5subtract_global_p50.avi
frame 0/68
frame 50/68
Wrote: exp148_roi5_subtract_tile_p50.avi


In [62]:
import numpy as np
import cv2

from skimage import filters, exposure, morphology
from skimage.feature import blob_log


# =========================
# Display helpers
# =========================
def compute_global_scale(stack, p_low=1, p_high=99):
    x = stack.astype(np.float32)
    lo, hi = np.percentile(x, (p_low, p_high))
    return float(lo), float(hi)

def to_uint8_fixed(img2d, lo, hi):
    img = exposure.rescale_intensity(img2d.astype(np.float32),
                                     in_range=(float(lo), float(hi)),
                                     out_range=(0, 255))
    return img.astype(np.uint8)

def to_uint8_auto(img2d, p_low=1, p_high=99.7):
    img = img2d.astype(np.float32)
    lo, hi = np.percentile(img, (p_low, p_high))
    lo, hi = float(lo), float(hi)
    if hi <= lo:
        return np.zeros_like(img2d, dtype=np.uint8)
    return to_uint8_fixed(img, lo, hi)

def draw_scale_bar(rgb, length_px=100, pad=12, thickness=4,
                   color=(255, 255, 255), loc="lower right"):
    H, W, _ = rgb.shape
    if loc == "lower right":
        x0 = W - pad - length_px
        y0 = H - pad
    elif loc == "lower left":
        x0 = pad
        y0 = H - pad
    else:
        raise ValueError("loc must be 'lower right' or 'lower left'")
    x1 = x0 + length_px
    y1 = y0 - thickness
    rgb[y1:y0, x0:x1] = color

def put_label(rgb, text, x=10, y=28):
    cv2.putText(rgb, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.72,
                (255, 255, 255), 2, cv2.LINE_AA)

def draw_box(rgb, bbox, color=(255, 0, 0), thickness=2):
    minr, minc, maxr, maxc = bbox
    minr, minc, maxr, maxc = map(int, [minr, minc, maxr, maxc])
    cv2.rectangle(rgb, (minc, minr), (maxc, maxr), color, thickness)

def draw_circle(rgb, y, x, r, color=(0, 255, 255), thickness=2):
    cv2.circle(rgb, (int(x), int(y)), int(r), color, thickness)


# =========================
# Step 1: tilewise subtract bottom percentile (your good visual step)
# =========================
def subtract_bottom_percentile_tilewise(g, tile=64, p=50, smooth_sigma_tiles=1.2, clamp=True):
    g = g.astype(np.float32)
    H, W = g.shape

    padH = (tile - (H % tile)) % tile
    padW = (tile - (W % tile)) % tile
    gp = np.pad(g, ((0, padH), (0, padW)), mode="reflect")
    Hp, Wp = gp.shape
    nH, nW = Hp // tile, Wp // tile

    base = np.zeros((nH, nW), dtype=np.float32)
    for i in range(nH):
        r0, r1 = i * tile, (i + 1) * tile
        for j in range(nW):
            c0, c1 = j * tile, (j + 1) * tile
            base[i, j] = np.percentile(gp[r0:r1, c0:c1], float(p))

    if smooth_sigma_tiles and smooth_sigma_tiles > 0:
        base = filters.gaussian(base, sigma=float(smooth_sigma_tiles), preserve_range=True)

    base_up = np.kron(base, np.ones((tile, tile), dtype=np.float32))[:Hp, :Wp]
    base_up = base_up[:H, :W]

    out = g - base_up
    if clamp:
        out[out < 0] = 0
    return out, base_up


# =========================
# Step 2: filament/vesselness mask (THIS is the real improvement)
# =========================
def filament_mask_vesselness(g, sigmas=(2, 3, 4, 5, 6), mask_percentile=99.2,
                             dilate=2, close=2):
    """
    Returns a boolean mask of filament-like structures using Frangi vesselness.
    mask_percentile: higher => fewer pixels marked as filaments.
    """
    g = g.astype(np.float32)

    # normalize to [0,1] for stable vesselness
    p1, p99 = np.percentile(g, (1, 99))
    p1, p99 = float(p1), float(p99)
    if p99 <= p1:
        g01 = np.zeros_like(g, dtype=np.float32)
    else:
        g01 = exposure.rescale_intensity(g, in_range=(p1, p99), out_range=(0.0, 1.0)).astype(np.float32)

    # vesselness
    v = filters.frangi(g01, sigmas=tuple(sigmas), black_ridges=False)
    v = v.astype(np.float32)

    thr = float(np.percentile(v, float(mask_percentile)))
    m = v > thr

    if close and close > 0:
        m = morphology.binary_closing(m, morphology.disk(int(close)))
    if dilate and dilate > 0:
        m = morphology.binary_dilation(m, morphology.disk(int(dilate)))

    return m, v


# =========================
# Step 3: blob detection in size range + reject blobs on filaments
# =========================
def detect_puncta_blobs(g, fil_mask=None,
                        min_diam=10, max_diam=50,
                        blob_threshold=0.06,
                        overlap=0.5,
                        reject_if_on_filament=True,
                        filament_reject_frac=0.20):
    """
    blob_log uses sigma ~ radius/sqrt(2). We map diameter -> radius -> sigma.
    """
    g = g.astype(np.float32)

    # normalize to [0,1] for stable blob thresholding
    p1, p99 = np.percentile(g, (1, 99))
    p1, p99 = float(p1), float(p99)
    if p99 <= p1:
        g01 = np.zeros_like(g, dtype=np.float32)
    else:
        g01 = exposure.rescale_intensity(g, in_range=(p1, p99), out_range=(0.0, 1.0)).astype(np.float32)

    rmin = min_diam / 2.0
    rmax = max_diam / 2.0
    min_sigma = rmin / np.sqrt(2.0)
    max_sigma = rmax / np.sqrt(2.0)

    blobs = blob_log(
        g01,
        min_sigma=float(min_sigma),
        max_sigma=float(max_sigma),
        num_sigma=10,
        threshold=float(blob_threshold),
        overlap=float(overlap)
    )  # rows: (y, x, sigma)

    kept = []
    for (y, x, s) in blobs:
        r = float(s * np.sqrt(2.0))  # approx radius

        if reject_if_on_filament and fil_mask is not None:
            # check fraction of filament pixels within blob circle (approx via bounding box)
            y0 = max(0, int(y - r))
            y1 = min(g.shape[0], int(y + r) + 1)
            x0 = max(0, int(x - r))
            x1 = min(g.shape[1], int(x + r) + 1)
            patch = fil_mask[y0:y1, x0:x1]
            if patch.size > 0:
                if (patch.mean() >= filament_reject_frac):
                    continue

        kept.append((float(y), float(x), float(r)))

    return kept


# =========================
# Video maker:
# Left = RAW (fixed)
# Right = PREPROCESSED (tile subtract) with filaments suppressed + puncta detections
# =========================
def make_video_tile_subtract_then_filament_suppress_then_blobs(
    green_stack,
    out_path="exp148_tileSub_filMask_blobs.avi",
    fps=10, k_start=0, k_end=None, stride=1, resize=1.0,
    scale_bar_px=100,
    raw_p_low=1, raw_p_high=99,

    # tile subtract params
    tile=64, bottom_p=50, smooth_sigma_tiles=1.2,

    # filament mask params
    frangi_sigmas=(2, 3, 4, 5, 6),
    fil_mask_percentile=99.2,
    fil_dilate=2,
    fil_close=2,
    suppress_strength=1.0,  # 1.0 => hard zero out filaments, 0.5 => reduce by half

    # blob params (puncta size)
    min_diam=10,
    max_diam=50,
    blob_threshold=0.06,
    blob_overlap=0.5,
    filament_reject_frac=0.20,

    # right panel view
    right_view="suppressed"  # "pre" | "vesselness" | "filmask" | "suppressed"
):
    H, W, T = green_stack.shape
    if k_end is None:
        k_end = T

    raw_lo, raw_hi = compute_global_scale(green_stack[:, :, k_start:k_end:stride], raw_p_low, raw_p_high)

    outW, outH = int(W * resize), int(H * resize)
    outW2 = outW * 2

    fourcc = cv2.VideoWriter_fourcc(*"MJPG")
    writer = cv2.VideoWriter(out_path, fourcc, fps, (outW2, outH), True)
    if not writer.isOpened():
        raise RuntimeError("Could not open VideoWriter.")

    for k in range(k_start, k_end, stride):
        raw = green_stack[:, :, k].astype(np.float32)

        # 1) tile subtract
        pre, _ = subtract_bottom_percentile_tilewise(
            raw, tile=tile, p=bottom_p, smooth_sigma_tiles=smooth_sigma_tiles, clamp=True
        )

        # 2) filament mask (on pre)
        fil_mask, vessel = filament_mask_vesselness(
            pre,
            sigmas=frangi_sigmas,
            mask_percentile=fil_mask_percentile,
            dilate=fil_dilate,
            close=fil_close
        )

        # 3) suppress filaments
        sup = pre.copy()
        sup[fil_mask] *= float(max(0.0, 1.0 - suppress_strength))  # 1.0 -> 0, 0.5 -> 0.5
        # extra: light blur to remove 1px speckle without killing blobs
        sup = filters.gaussian(sup, sigma=0.7, preserve_range=True).astype(np.float32)

        # 4) blobs on suppressed image, reject on filaments
        blobs = detect_puncta_blobs(
            sup, fil_mask=fil_mask,
            min_diam=min_diam, max_diam=max_diam,
            blob_threshold=blob_threshold,
            overlap=blob_overlap,
            reject_if_on_filament=True,
            filament_reject_frac=filament_reject_frac
        )

        # LEFT render: raw fixed
        raw8 = to_uint8_fixed(raw, raw_lo, raw_hi)
        left = np.stack([raw8, raw8, raw8], axis=-1)
        draw_scale_bar(left, length_px=scale_bar_px)
        put_label(left, f"RAW fixed  k={k}", x=10, y=28)

        # RIGHT render: chosen view
        if right_view == "pre":
            right_img = pre
            title = f"tileSub p{bottom_p} tile={tile}"
            right8 = to_uint8_auto(right_img)
            right = np.stack([right8, right8, right8], axis=-1)

        elif right_view == "vesselness":
            right_img = vessel
            title = f"vesselness (Frangi) p{fil_mask_percentile}"
            right8 = to_uint8_auto(right_img, 1, 99.5)
            right = np.stack([right8, right8, right8], axis=-1)

        elif right_view == "filmask":
            right8 = (fil_mask.astype(np.uint8) * 255)
            title = "filament mask"
            right = np.stack([right8, right8, right8], axis=-1)

        elif right_view == "suppressed":
            right_img = sup
            title = f"suppressed + blobs n={len(blobs)}"
            right8 = to_uint8_auto(right_img)
            right = np.stack([right8, right8, right8], axis=-1)

        else:
            raise ValueError("right_view must be one of: pre, vesselness, filmask, suppressed")

        # overlay blobs on RIGHT
        for (y, x, r) in blobs:
            draw_circle(right, y, x, r, color=(0, 255, 255), thickness=2)

        draw_scale_bar(right, length_px=scale_bar_px)
        put_label(right, title, x=10, y=28)

        frame = np.concatenate([left, right], axis=1)

        if resize != 1.0:
            frame = cv2.resize(frame, (outW2, outH), interpolation=cv2.INTER_AREA)

        writer.write(frame[:, :, ::-1])  # RGB->BGR

        if (k - k_start) % (25 * stride) == 0:
            print(f"frame {k}/{k_end-1} blobs={len(blobs)}")

    writer.release()
    print("Wrote:", out_path)


# =========================
# Run
# =========================
if __name__ == "__main__":
    data = np.load("./stacks_exp_148/exp_148_roi_1_stack.npz")
    green_stack = data["Tstack"].astype(np.float32)  # (H,W,T)
    print("stack:", green_stack.shape, green_stack.dtype)

    make_video_tile_subtract_then_filament_suppress_then_blobs(
        green_stack,
        out_path="exp148_tileSub_filSuppress_blobs.avi",
        fps=10,
        stride=2,
        resize=0.5,
        scale_bar_px=100,

        # tile subtraction
        tile=64,
        bottom_p=50,
        smooth_sigma_tiles=1.2,

        # filament suppression
        frangi_sigmas=(2, 3, 4, 5, 6),
        fil_mask_percentile=99.2,  # increase -> fewer pixels classified as filaments (99.2 -> 99.5)
        fil_dilate=2,
        fil_close=2,
        suppress_strength=1.0,     # 1.0 hard remove filaments

        # blobs (10–50 px diam)
        min_diam=10,
        max_diam=50,
        blob_threshold=0.07,       # increase -> fewer detections (0.06 -> 0.09)
        blob_overlap=0.5,
        filament_reject_frac=0.20,

        right_view="suppressed"
    )


stack: (501, 501, 69) float32
frame 0/68 blobs=676
frame 50/68 blobs=117
Wrote: exp148_tileSub_filSuppress_blobs.avi


In [65]:
import numpy as np
import cv2
from skimage import exposure

def compute_global_scale(stack, p_low=1, p_high=99):
    x = stack.astype(np.float32)
    lo, hi = np.percentile(x, (p_low, p_high))
    return float(lo), float(hi)

def to_uint8_fixed(img2d, lo, hi):
    img = exposure.rescale_intensity(img2d.astype(np.float32),
                                     in_range=(float(lo), float(hi)),
                                     out_range=(0, 255))
    return img.astype(np.uint8)

def draw_scale_bar(rgb, length_px=100, pad=12, thickness=4,
                   color=(255, 255, 255), loc="lower right"):
    H, W, _ = rgb.shape
    if loc == "lower right":
        x0 = W - pad - length_px
        y0 = H - pad
    elif loc == "lower left":
        x0 = pad
        y0 = H - pad
    else:
        raise ValueError("loc must be 'lower right' or 'lower left'")
    x1 = x0 + length_px
    y1 = y0 - thickness
    rgb[y1:y0, x0:x1] = color

def put_label(rgb, text, x=10, y=28):
    cv2.putText(rgb, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.72,
                (255, 255, 255), 2, cv2.LINE_AA)

def make_green_plus_globalmax_red_video(
    green_stack,
    out_path="green_live_plus_red_globalmax.avi",
    fps=10,
    k_start=0,
    k_end=None,
    stride=1,
    resize=1.0,
    scale_bar_px=100,
    raw_p_low=1,
    raw_p_high=99,
    # thresholding on max projection
    hot_percentile=90.0,        # percentile on max-projection image
    red_alpha=1.0,              # 1.0 = solid red on hot pixels
    red_value=255               # how strong red is
):
    H, W, T = green_stack.shape
    if k_end is None:
        k_end = T

    # fixed scaling for the live frame (green channel), stable across time
    raw_lo, raw_hi = compute_global_scale(green_stack[:, :, k_start:k_end:stride], raw_p_low, raw_p_high)

    # global max across all times (or the chosen range)
    maxproj = np.max(green_stack[:, :, k_start:k_end:stride].astype(np.float32), axis=2)

    # threshold computed ONCE, global
    thr = float(np.percentile(maxproj, float(hot_percentile)))
    hot = maxproj >= thr  # static mask

    outW, outH = int(W * resize), int(H * resize)

    fourcc = cv2.VideoWriter_fourcc(*"MJPG")
    writer = cv2.VideoWriter(out_path, fourcc, fps, (outW, outH), True)
    if not writer.isOpened():
        raise RuntimeError("Could not open VideoWriter (try .avi with MJPG).")

    for k in range(k_start, k_end, stride):
        g = green_stack[:, :, k].astype(np.float32)

        # live frame -> green channel
        g8 = to_uint8_fixed(g, raw_lo, raw_hi)

        # build RGB: start black, set green = live frame
        rgb = np.zeros((H, W, 3), dtype=np.float32)
        rgb[..., 1] = g8.astype(np.float32)  # green

        # red overlay from static hot mask (from max projection)
        if red_alpha >= 1.0:
            rgb[hot, 0] = float(red_value)
        else:
            rgb[hot, 0] = (1 - red_alpha) * rgb[hot, 0] + red_alpha * float(red_value)

        rgb = np.clip(rgb, 0, 255).astype(np.uint8)

        draw_scale_bar(rgb, length_px=scale_bar_px)
        put_label(rgb, f"GREEN=frame k={k} | RED=maxproj >= p{hot_percentile:.0f} (thr={thr:.1f})", x=10, y=28)

        if resize != 1.0:
            rgb = cv2.resize(rgb, (outW, outH), interpolation=cv2.INTER_AREA)

        writer.write(rgb[:, :, ::-1])  # RGB -> BGR

        if (k - k_start) % (25 * stride) == 0:
            print(f"frame {k}/{k_end-1}")

    writer.release()
    print("Wrote:", out_path)

    # also save the max projection as a PNG for inspection
    max8 = to_uint8_fixed(maxproj, *compute_global_scale(maxproj[..., None], 1, 99))
    cv2.imwrite(out_path.replace(".avi", "_maxproj.png"), max8)
    print("Saved max projection image:", out_path.replace(".avi", "_maxproj.png"))


# ---- run ----
if __name__ == "__main__":
    data = np.load("./stacks_exp_148/exp_148_roi_1_stack.npz")
    green_stack = data["Tstack"].astype(np.float32)  # (H,W,T)

    make_green_plus_globalmax_red_video(
        green_stack,
        out_path="exp148_green_live_plus_red_maxproj_p90.avi",
        fps=4,
        stride=2,
        resize=0.5,
        hot_percentile=80.0,   # try 95 or 97 if too much red
        red_alpha=0.5,
        red_value=255
    )


frame 0/68
frame 50/68
Wrote: exp148_green_live_plus_red_maxproj_p90.avi
Saved max projection image: exp148_green_live_plus_red_maxproj_p90_maxproj.png


In [ ]:
import numpy as np
import cv2
from skimage import filters, exposure, morphology, measure


# =========================
# Display helpers (BGR)
# =========================
def compute_global_scale(stack, p_low=1, p_high=99):
    x = stack.astype(np.float32)
    lo, hi = np.percentile(x, (p_low, p_high))
    return float(lo), float(hi)

def to_uint8_fixed(img2d, lo, hi):
    img = exposure.rescale_intensity(img2d.astype(np.float32),
                                     in_range=(float(lo), float(hi)),
                                     out_range=(0, 255))
    return img.astype(np.uint8)

def draw_rect_bgr(bgr, minr, minc, maxr, maxc, thickness=2, color=(0, 0, 255)):
    # color is BGR
    H, W, _ = bgr.shape
    minr = max(0, min(H - 1, int(minr)))
    maxr = max(0, min(H - 1, int(maxr)))
    minc = max(0, min(W - 1, int(minc)))
    maxc = max(0, min(W - 1, int(maxc)))
    if maxr <= minr or maxc <= minc:
        return
    cv2.rectangle(bgr, (minc, minr), (maxc, maxr), color, int(thickness))

def draw_scale_bar_bgr(bgr, length_px=100, pad=12, thickness=4,
                       color=(255, 255, 255), loc="lower right"):
    H, W, _ = bgr.shape
    if loc == "lower right":
        x0 = W - pad - length_px
        y0 = H - pad
    elif loc == "lower left":
        x0 = pad
        y0 = H - pad
    else:
        raise ValueError("loc must be 'lower right' or 'lower left'")
    x1 = x0 + length_px
    y1 = y0 - thickness
    bgr[y1:y0, x0:x1] = color

def put_label_bgr(bgr, text, x=10, y=28):
    # Ensure contiguous uint8
    if bgr.dtype != np.uint8:
        bgr[:] = np.clip(bgr, 0, 255).astype(np.uint8)
    if not bgr.flags["C_CONTIGUOUS"]:
        bgr[:] = np.ascontiguousarray(bgr)
    cv2.putText(bgr, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.72,
                (255, 255, 255), 2, cv2.LINE_AA)


# =========================
# Static mask from global max projection
# =========================
def global_max_mask(stack, k_start=0, k_end=None, stride=1, hot_percentile=90.0):
    H, W, T = stack.shape
    if k_end is None:
        k_end = T
    maxproj = np.max(stack[:, :, k_start:k_end:stride].astype(np.float32), axis=2)
    thr = float(np.percentile(maxproj, float(hot_percentile)))
    mask = maxproj >= thr
    return mask, thr, maxproj


# =========================
# Your ORIGINAL detector (Gaussian -> CLAHE -> Otsu -> cleanup -> CC)
# =========================
def detect_original_pipeline(
    g,
    min_area=80,
    blur_sigma=1.2,
    clahe_clip=0.01,
    close_disk=3,
):
    g = g.astype(np.float32)

    g_blur = filters.gaussian(g, sigma=float(blur_sigma), preserve_range=True)

    # CLAHE expects float in [0,1]
    p1, p99 = np.percentile(g_blur, (1, 99))
    p1, p99 = float(p1), float(p99)
    if p99 <= p1:
        g01 = np.zeros_like(g_blur, dtype=np.float32)
    else:
        g01 = exposure.rescale_intensity(g_blur, in_range=(p1, p99), out_range=(0.0, 1.0)).astype(np.float32)

    g_eq = exposure.equalize_adapthist(g01, clip_limit=float(clahe_clip))

    thr = filters.threshold_otsu(g_eq)
    mask = g_eq > thr

    mask = morphology.remove_small_objects(mask, min_size=int(min_area))
    mask = morphology.binary_closing(mask, morphology.disk(int(close_disk)))

    lab = measure.label(mask)
    props = measure.regionprops(lab)
    boxes = [p.bbox for p in props]
    return boxes, lab, props


# =========================
# Restrict detections by overlap with global mask
# =========================
def filter_boxes_by_mask(lab, props, global_mask, agree_frac=0.20):
    kept_boxes = []
    for p in props:
        minr, minc, maxr, maxc = p.bbox
        region = (lab[minr:maxr, minc:maxc] == p.label)
        gm = global_mask[minr:maxr, minc:maxc]
        if region.sum() == 0:
            continue
        frac = float((region & gm).sum()) / float(region.sum())
        if frac >= float(agree_frac):
            kept_boxes.append(p.bbox)
    return kept_boxes


# =========================
# Video:
# Left  = RAW + all original detections (BLUE boxes)
# Right = RAW + global max mask overlay (RED pixels) + kept detections (RED boxes)
# =========================
def make_restricted_detection_video(
    green_stack,
    out_path="orig_restricted_by_globalmax.avi",
    fps=10,
    k_start=0,
    k_end=None,
    stride=1,
    resize=1.0,
    scale_bar_px=100,
    raw_p_low=1,
    raw_p_high=99,

    hot_percentile=90.0,

    min_area=80,
    blur_sigma=1.2,
    clahe_clip=0.01,
    close_disk=3,

    agree_frac=0.20,
    mask_alpha=0.55
):
    H, W, T = green_stack.shape
    if k_end is None:
        k_end = T

    raw_lo, raw_hi = compute_global_scale(green_stack[:, :, k_start:k_end:stride], raw_p_low, raw_p_high)

    gm, gm_thr, _ = global_max_mask(green_stack, k_start=k_start, k_end=k_end, stride=stride,
                                   hot_percentile=hot_percentile)

    outW, outH = int(W * resize), int(H * resize)
    outW2 = outW * 2

    fourcc = cv2.VideoWriter_fourcc(*"MJPG")
    writer = cv2.VideoWriter(out_path, fourcc, fps, (outW2, outH), True)
    if not writer.isOpened():
        raise RuntimeError("Could not open VideoWriter.")

    for k in range(k_start, k_end, stride):
        raw = green_stack[:, :, k].astype(np.float32)

        boxes_all, lab, props = detect_original_pipeline(
            raw,
            min_area=min_area,
            blur_sigma=blur_sigma,
            clahe_clip=clahe_clip,
            close_disk=close_disk
        )

        boxes_kept = filter_boxes_by_mask(lab, props, gm, agree_frac=agree_frac)

        # base grayscale -> BGR
        raw8 = to_uint8_fixed(raw, raw_lo, raw_hi)
        base_bgr = cv2.cvtColor(raw8, cv2.COLOR_GRAY2BGR)

        # LEFT: all detections in BLUE (BGR blue = (255,0,0))
        left = base_bgr.copy()
        for (minr, minc, maxr, maxc) in boxes_all:
            draw_rect_bgr(left, minr, minc, maxr, maxc, thickness=2, color=(255, 0, 0))

        # RIGHT: mask overlay in RED + kept detections in RED
        right = base_bgr.copy().astype(np.float32)

        # overlay red pixels where gm True: BGR red channel is index 2
        right[gm, 2] = (1 - mask_alpha) * right[gm, 2] + mask_alpha * 255.0
        right[gm, 1] = (1 - mask_alpha) * right[gm, 1]
        right[gm, 0] = (1 - mask_alpha) * right[gm, 0]

        right = np.clip(right, 0, 255).astype(np.uint8)

        for (minr, minc, maxr, maxc) in boxes_kept:
            draw_rect_bgr(right, minr, minc, maxr, maxc, thickness=2, color=(0, 0, 255))

        draw_scale_bar_bgr(left, length_px=scale_bar_px)
        draw_scale_bar_bgr(right, length_px=scale_bar_px)

        put_label_bgr(left,  f"RAW + ORIGINAL boxes n={len(boxes_all)}  k={k}", x=10, y=28)
        put_label_bgr(right, f"Restricted by max-mask p{hot_percentile:.0f} thr={gm_thr:.1f} | kept n={len(boxes_kept)}",
                      x=10, y=28)

        frame = np.concatenate([left, right], axis=1)

        if resize != 1.0:
            frame = cv2.resize(frame, (outW2, outH), interpolation=cv2.INTER_AREA)

        writer.write(frame)

        if (k - k_start) % (25 * stride) == 0:
            print(f"frame {k}/{k_end-1} all={len(boxes_all)} kept={len(boxes_kept)}")

    writer.release()
    print("Wrote:", out_path)


# =========================
# Example run
# =========================
if __name__ == "__main__":
    data = np.load("./stacks_exp_148/exp_148_roi_9_stack.npz")
    green_stack = data["Tstack"].astype(np.float32)  # (H,W,T)

    make_restricted_detection_video(
        green_stack,
        out_path="exp148_original_restricted_by_globalmax.avi",
        fps=4,
        stride=2,
        resize=0.5,
        scale_bar_px=100,

        hot_percentile=95.0,   # tighten mask: 90 -> 95 -> 97
        agree_frac=0.4,       # stricter agreement: 0.2 -> 0.4 -> 0.6

        min_area=80,
        blur_sigma=1.2,
        clahe_clip=0.01,
        close_disk=3,

        mask_alpha=0.0
    )


frame 0/68 all=59 kept=12
frame 50/68 all=21 kept=17
Wrote: exp148_original_restricted_by_globalmax.avi
